# DP-SGD against its two ℓ₁-projected counterparts, on Criteo

Notebook 01 ran DP-SGD end to end on Criteo and showed the pieces work.
Notebook 02 gave it a non-private baseline and priced the privacy. Both left
the optimizer seam alone: every run descended along `updates.sgd(lr)` and
nothing stood between the mechanism's released estimate and the parameters.

Two wrappers now sit at that seam. `l1_projected` constrains the **iterates** —
every step lands inside one global ℓ₁ ball. `l1_projected_estimate` constrains
the **estimate** — the released gradient is projected into the ball and the
rule descends along that instead. They arrived in PRs #21 and #24 (ADR-0014,
ADR-0015) with Criteo evidence in the PR bodies, produced by demo scripts that
were never committed. That evidence is therefore unreproducible from the
repository as it stands, which is the gap this notebook closes: the same
comparison, in a file that runs, at a fixed budget of **ε = 3, δ = 1e-6**.

The finding is two-sided, and the second side is a negative result reported as
one. On the iterate side the ball binds, costs a little log-loss and zeroes
weights. On the estimate side the projection is a **no-op at every principled
radius** — the released estimate's ℓ₁ norm is already two orders of magnitude
inside the ball the paper prescribes — and even forced far below that it moves
log-loss by 0.0001. ADR-0015 predicts exactly this, and section 4 measures the
reason rather than asserting it.

**On the numbering.** Notebook `03-what-the-feature-norm-bound-costs` exists on
its own branch, unmerged, and is cited by the notebook-map issue #4, so this
one is **04**. The comparison against Private SpiderBoost is issue #11's
notebook and is not here.

**On the step count.** 2,000 steps, where notebook 02 ran 10,000. The purpose
of this notebook is to reproduce PR #21's and PR #24's tables digit-for-digit,
and those runs were 2,000 steps; changing the length would compare against
nothing. σ is recalibrated for 2,000 steps below, so the budget is honoured at
the shorter length — **this is a reproduction budget, not a convergence
budget**, and section 6's sweep is the only place a longer run would plausibly
change the shape of a conclusion.

What is fixed across every arm, and why (ADR-0005, ADR-0002):

- the same per-sample loss, `per_sample_bce_loss`;
- the same model and the **same initialization**, `init_params(jax.random.key(0), d)`;
- the same data, the same split, the same evaluation set, the same threshold rule;
- the same seeds — noise `jax.random.key(1)`, sampling `np.random.default_rng(2)`,
  drawn fresh for every arm;
- the same 2,000 optimizer steps and the same σ, so every arm carries the
  identical ε.

**The one thing that differs between arms is the `updates.Optimizer` object
handed to `train`.** That is enforced below by a single keyword-only runner
which takes the optimizer and nothing else.

Imports and the constants the whole notebook runs on. Everything that a run
depends on is named here, before any of it is used.

In [1]:
import time
from typing import NamedTuple

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from dimma.accounting.sampling import (
    PoissonGaussianSchedule,
    calibrate_noise_multiplier,
    poisson_gaussian_epsilon,
)
from dimma.algorithms.dp_sgd import train as dp_sgd
from dimma.core import pytree, updates
from dimma.datasets.criteo import load_criteo
from dimma.metrics.calibration import (
    calibration_ratio,
    expected_calibration_error,
)
from dimma.metrics.scoring import log_loss, normalized_entropy
from dimma.models.logreg import forward, init_params
from dimma.models.losses import per_sample_bce_loss
from dimma.transforms.projection import l1_projected, l1_projected_estimate

# The budget, fixed before anything is fitted.
TARGET_EPSILON = 3.0
TARGET_DELTA = 1e-6

# The step budget. Every arm takes exactly this many optimizer steps, and this
# is the composition count sigma is calibrated against. 2,000 rather than
# notebook 02's 10,000: this notebook reproduces PR #21/#24.
STEPS = 2_000

# Algorithm 1's L, C and eta. Held identical across every arm.
EXPECTED_BATCH_SIZE = 1024
CLIP_NORM = 5.0
LEARNING_RATE = 0.1

N_BINS = 15    # notebook 01's bin count, so the ECEs compare across notebooks

# Three streams, three seeds. The initialization is shared by every arm; the
# noise and sampling streams are re-created fresh for each arm so that two arms
# see the same draws.
INIT_SEED, NOISE_SEED, SAMPLING_SEED = 0, 1, 2

DEVICE = "gpu" if any(d.platform == "gpu" for d in jax.devices()) else "cpu"
print(f"jax devices: {jax.devices()}  ->  DEVICE = {DEVICE!r}")

jax devices: [CudaDevice(id=0)]  ->  DEVICE = 'gpu'


### Chart styling

One place for the palette, so every figure below reads as the same system.
Three categorical slots: **control** blue, **iterate-projected** orange,
**estimate-projected** aqua. The triple clears the colour-vision-deficiency
and normal-vision separation floors against a white surface (worst all-pairs
CVD ΔE 9.2).

The aqua sits at 2.74:1 against white, which is under the 3:1 non-text
contrast floor. It is used anyway, under the relief rule: **every series it
draws is also printed as a number in a table**, in sections 5 and 6, so no
reading depends on seeing that line. Hue never carries a series on its own —
every multi-series figure has a legend.

In [ ]:
SERIES = {
    "control": "#2a78d6",             # categorical slot 1
    "iterate-projected": "#eb6834",   # slot 2
    "estimate-projected": "#1baf7a",  # slot 3
}
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": MUTED,
    "axes.titlecolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "grid.color": GRID, "grid.linewidth": 0.8,
    "lines.linewidth": 2.0, "lines.markersize": 6.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "font.size": 10,
})

## 1. The data

`load_criteo` is the only thing here that touches the dataset, and it records
what it did in `metadata["preprocessing"]` so a run carries its own provenance
(ADR-0008). All 39 features, standardized, on the deterministic `seed=0` split.

**No `feature_norm_bound`.** Notebook 02 enforced `R = 2.0` because a
comparison against a non-private baseline needs the same data in both arms and
`R` is the parameter that pins it (ADR-0012). This notebook compares DP-SGD
against itself with a wrapper added, so every arm already sees byte-identical
data whatever `R` is, and the PR runs being reproduced did not set one.
`feature_norm_bound=None` is passed explicitly rather than left to the default,
so the choice is visible. The consequence: **these numbers are not comparable
to notebook 02's**, which are computed on capped rows.

In [3]:
split = load_criteo(features="all", preprocess=True, standardize=True,
                    download=False, seed=0, device=DEVICE,
                    feature_norm_bound=None)

n_train, n_features = split.x_train.shape
D_TOTAL = n_features + 1          # 39 weights + the bias, counted globally

X_TEST = split.x_test
Y_TEST = np.asarray(split.y_test, dtype=np.float64)

train_base_rate = float(split.y_train.mean())
test_base_rate = float(split.y_test.mean())

print(f"train {split.x_train.shape}   test {split.x_test.shape}")
print(f"parameters: {n_features} weights + 1 bias = {D_TOTAL}")
print(f"label base rate: {train_base_rate:.4f} train, "
      f"{test_base_rate:.4f} test")
print(f"PR-AUC floor (a random ranking scores the base rate): "
      f"{test_base_rate:.4f}")
print(f"arrays live on: {list(X_TEST.devices())}")
print()
# The key is written only when a bound was actually enforced, so its
# absence is the record that none was - see the note above.
print(f"enforced feature norm bound R = "
      f"{split.metadata.get('feature_norm_bound', 'none enforced')}")
print()
print(split.metadata["preprocessing"])

criteo: 39 features. I1-I13: NaN filled with the train-split median, clipped below at 0, log1p — the fill and the clip are no-ops on stored values that carry neither NaN nor negatives, and log1p acts on [0, 1]. C1-C26: each category ID replaced by that category's relative frequency in the train split, with categories unseen in training encoded as 0.0. Every column then standardized by the train-split mean and standard deviation.


train (800000, 39)   test (200000, 39)
parameters: 39 weights + 1 bias = 40
label base rate: 0.2510 train, 0.2520 test
PR-AUC floor (a random ranking scores the base rate): 0.2520
arrays live on: [CudaDevice(id=0)]

enforced feature norm bound R = none enforced

39 features. I1-I13: NaN filled with the train-split median, clipped below at 0, log1p — the fill and the clip are no-ops on stored values that carry neither NaN nor negatives, and log1p acts on [0, 1]. C1-C26: each category ID replaced by that category's relative frequency in the train split, with categories unseen in training encoded as 0.0. Every column then standardized by the train-split mean and standard deviation.


### The caveats that travel with every ε below

Stated here once, and restated in the closing section, rather than filed away
(ADR-0008, and the list in `accounting/sampling.py`):

1. **The preprocessing statistics** — the medians, the category frequencies,
   the means and standard deviations — are fitted on the training split and
   accounted for in no budget.
2. **The ε covers the training loop only.** Nothing in this notebook is tuned:
   `C`, `η` and `L` are the PR runs' values, taken as given, so there is no
   unaccounted hyperparameter search. The radii in section 3 are the one thing
   read off a trained model, and that read is discussed where it happens.
3. **Evaluating on test is outside the loop by design.** `train` returns
   parameters and no metrics, precisely so the loop is not in a position to
   leak them.

## 2. Calibrating for 2,000 steps

`calibrate_noise_multiplier` runs the accountant backwards: given a target
`(ε, δ)` and the shape of the run, it returns the smallest noise multiplier
meeting the budget. It takes a *builder* — candidate multiplier to the release
schedules a run would have at that multiplier — because a method may release on
more than one schedule. Plain DP-SGD has exactly one, so the builder returns a
single-entry list. `poisson_gaussian_epsilon` then runs the same accountant
forwards to price the run at the multiplier that came back, and the two are
asserted consistent.

`T` is the load-bearing constraint. **The step count the calibration assumes is
the step count every arm actually takes.** Calibrating at 10,000 and training
for 2,000 would over-noise; calibrating at 2,000 and training for 10,000 would
report an ε the run does not have. Because `T` is 2,000 here and not notebook
02's 10,000, σ is *smaller* than notebook 02's at the same budget — fewer
compositions, less noise needed per step.

`clip_norm` does **not** appear. Sensitivity is `S = C` by construction and the
noise scales with it (`σ·C`), so the multiplier is the same at every `C`.

**The ε is RDP's** (ADR-0011, the `method="rdp"` default). PLD would report a
smaller number for the same run. The method is held fixed at `"rdp"`
everywhere in this notebook, which is the property a comparison needs.

In [4]:
q = EXPECTED_BATCH_SIZE / n_train

t0 = time.time()
SIGMA = calibrate_noise_multiplier(
    lambda multiplier: [PoissonGaussianSchedule(q, multiplier, STEPS)],
    TARGET_EPSILON, TARGET_DELTA,
)
spent = poisson_gaussian_epsilon(q, SIGMA, STEPS, TARGET_DELTA)
assert spent <= TARGET_EPSILON + 1e-9

print(f"q = L / n = {EXPECTED_BATCH_SIZE} / {n_train:,} = {q:.6f}")
print(f"sigma = {SIGMA:.6f}")
print(f"epsilon spent = {spent:.6f} at delta = {TARGET_DELTA} "
      f"over {STEPS:,} steps  (method='rdp')")
print(f"\ncalibrated in {time.time() - t0:.1f}s")

q = L / n = 1024 / 800,000 = 0.001280
sigma = 0.638442
epsilon spent = 2.999997 at delta = 1e-06 over 2,000 steps  (method='rdp')

calibrated in 0.2s


### Reading the calibration

σ = 0.638442 at q = 0.001280 over 2,000 steps buys ε = 2.999997 — the search
returns the smallest multiplier meeting the budget, so ε lands just under 3
rather than on it, and the assertion is what checks that. Every arm below runs
at this σ and therefore carries this ε, unchanged: **a projection applied after
the mechanism releases its estimate is post-processing and cannot increase the
cost.** That is the only privacy claim in this notebook, and it is the standard
one.

## 3. The control, and where the radii come from

The runner is keyword-only and takes the optimizer alone. Everything else —
initialization, data, both seeds, `STEPS`, `L`, `C`, σ — is closed over, so
"the arms differ only in the optimizer" is a property of the code rather than a
claim in prose.

The metric set is notebook 01's, condensed:

- **log-loss**, the objective every arm descends, so training and reporting
  agree on what better means;
- **normalized entropy** and the **calibration ratio** — log-loss over the base
  rate's entropy, and observed clicks over predicted clicks;
- **ECE** at 15 equal-mass bins, because per-example clipping is the documented
  cause of miscalibration under DP;
- **PR-AUC**, the ranking read, floored at the base rate. It stays a
  notebook-local helper: `dimma.metrics` offers no metric that reads rank alone
  and `tests/metrics/test_package_surface.py` pins that absence, so `pr_curve`
  is local **by decision, not by omission**. ROC-AUC is not reported.

**The threshold is a choice, not a discovery.** Predictions are `probs >
THRESHOLD` with `THRESHOLD` the *training-split* base rate — notebook 01's
rule, and the rule that produced the PR tables being reproduced. It is
strictly-greater, which matters for exactly one row: a constant predictor whose
output equals the threshold predicts nothing, and its precision is undefined
(`tp + fp == 0`). That row prints `0.0000`, as the PR tables did.

`‖θ‖₁` and the non-zero count are taken over the **ravelled pytree** — weights
and bias together, 40 numbers — because that is the object the transforms
project: one global ball across all leaves, not one per leaf.

In [ ]:
class Eval(NamedTuple):
    """What one arm is scored on. All on the test split."""

    loss: float
    ne: float
    ece: float
    cal: float
    pr_auc: float
    precision: float
    recall: float
    l1: float
    nnz: int


THRESHOLD = train_base_rate


def flat(params):
    """The parameters as one vector, in the order the transforms see them."""
    return np.concatenate(
        [np.asarray(v).ravel() for v in jax.tree_util.tree_leaves(params)]
    )


def l1_norm(params) -> float:
    return float(np.abs(flat(params)).sum())


def nnz(params) -> int:
    return int((flat(params) != 0.0).sum())


def l1_norm_traced(tree):
    """`l1_norm` for a pytree of tracers, for the recorder in section 4."""
    return sum(jnp.abs(v).sum() for v in jax.tree_util.tree_leaves(tree))


def pr_curve(probs, y):
    """Precision and recall at every cut, plus average precision (PR-AUC).

    Sort by score descending and integrate precision over recall at every
    positive. Local by decision rather than by omission - see above.
    """
    order = np.argsort(-probs)
    hits = y[order]
    positives = hits.sum()
    tp = np.cumsum(hits)
    precision = tp / np.arange(1, len(hits) + 1)
    recall = tp / positives
    pr_auc = float((precision * hits).sum() / positives)
    return precision, recall, pr_auc


def probabilities_of(params):
    """Test-split probabilities. `forward` is a single-example function."""
    logits = np.asarray(jax.vmap(forward, in_axes=(None, 0))(params, X_TEST))
    return np.asarray(jax.nn.sigmoid(jnp.asarray(logits)), dtype=np.float64)


def score(probs, params) -> Eval:
    """Score one set of test probabilities at the fixed threshold."""
    pred = probs > THRESHOLD
    truth = Y_TEST > 0.5
    tp = int((pred & truth).sum())
    fp = int((pred & ~truth).sum())
    fn = int((~pred & truth).sum())
    return Eval(
        loss=log_loss(probs, Y_TEST),
        ne=normalized_entropy(probs, Y_TEST),
        ece=expected_calibration_error(probs, Y_TEST, n_bins=N_BINS),
        cal=calibration_ratio(probs, Y_TEST),
        pr_auc=pr_curve(probs, Y_TEST)[2],
        # undefined precision prints as 0.0000; see the note above
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        l1=l1_norm(params) if params is not None else 0.0,
        nnz=nnz(params) if params is not None else 0,
    )


def evaluate(params) -> Eval:
    return score(probabilities_of(params), params)


def run_arm(*, optimizer):
    """One arm. The optimizer is the only thing a caller may vary."""
    params = init_params(jax.random.key(INIT_SEED), n_features)
    return dp_sgd.train(
        per_sample_bce_loss, params, optimizer,
        split.x_train, split.y_train,
        jax.random.key(NOISE_SEED), np.random.default_rng(SAMPLING_SEED),
        steps=STEPS, expected_batch_size=EXPECTED_BATCH_SIZE,
        clip_norm=CLIP_NORM, noise_multiplier=SIGMA,
    )

The control is plain DP-SGD: `updates.sgd(0.1)`, nothing wrapped around it. It
is both the first row of section 5's table and the source of one of the radii,
which is the reason it runs on its own here.

Three radii come out of this cell, and they are three different kinds of
number:

- `RADIUS_ITER = 0.5 × ‖θ_control‖₁` — **read off the control**, deliberately
  half its norm so the ball is guaranteed to bind. This is the one quantity in
  the notebook derived from a trained model, and it is a choice about what to
  illustrate, not a tuned hyperparameter.
- `RADIUS_PAPER = C·√d` — Ghazi et al.'s prescription for records ℓ₂-bounded by
  `C` and `d`-sparse, evaluated at this problem's `C` and `d`. Fixed a priori.
- `RADIUS_BINDING = 0.19` — hardcoded, PR #24's value, chosen there to force
  the estimate-side projection to bind. Section 4 measures the quantity that
  justifies it.

In [ ]:
t0 = time.time()
control_params = run_arm(optimizer=updates.sgd(LEARNING_RATE))
CONTROL = evaluate(control_params)
print(f"control run: {time.time() - t0:.1f}s\n")

RADIUS_WIDE = 1e6                                   # effectively no constraint
RADIUS_ITER = 0.5 * CONTROL.l1                      # read off the control
RADIUS_PAPER = CLIP_NORM * np.sqrt(D_TOTAL)         # Ghazi et al.'s C*sqrt(d)
RADIUS_BINDING = 0.19                               # PR #24's, hardcoded

print(f"control test log-loss {CONTROL.loss:.4f}   PR-AUC {CONTROL.pr_auc:.4f}"
      f"   ECE {CONTROL.ece:.4f}")
print(f"control ||theta||_1 = {CONTROL.l1:.4f}   "
      f"non-zero {CONTROL.nnz} of {D_TOTAL}")
print()
print(f"RADIUS_WIDE    = {RADIUS_WIDE:g}       (no constraint, both sides)")
print(f"RADIUS_ITER    = {RADIUS_ITER:.4f}   (0.5 x control ||theta||_1)")
print(f"RADIUS_PAPER   = {RADIUS_PAPER:.4f}  (C x sqrt({D_TOTAL}))")
print(f"RADIUS_BINDING = {RADIUS_BINDING}      (PR #24's, forced to bind)")

### Reading the control

Test log-loss 0.5142 at ε = 3, against a constant predictor's 0.5645 (section
5's last row): the model has learned something, and at 40 parameters over
800,000 rows that is not surprising. All 40 parameters are non-zero, which is
what makes the non-zero count a meaningful column later — any zero below is
produced by a projection, not inherited.

`‖θ_control‖₁ = 5.2602` is the number the rest of the notebook is scaled
against. Note how far it sits from `RADIUS_PAPER = 31.62`: the *parameters* are
already six times inside the ball the paper prescribes for the *estimate*, and
section 4 shows the estimate is two orders of magnitude inside it.

## 4. What the mechanism actually releases

The estimate-side result in section 5 is a no-op, and a no-op is only worth
reporting if the reason is measured. The reason is the ℓ₁ norm of what DP-SGD
releases each step, against the radius the projection would impose.

That quantity is not returned by anything. `train` hands back parameters and
nothing else, by design. So it is instrumented at the seam with a wrapper that
records and does not touch:

**This wrapper is notebook-local, is post-processing, and makes no privacy
claim.** It also stays out of the library deliberately: ADR-0014's seam is a
caller-side layer for changing what an optimizer *does*, and a recorder does
not belong in it.

**One mechanical trap, stated because the code looks odd without it.** `train`
jits its step with the optimizer *baked into the traced function*, so a plain
Python `list.append` inside the wrapper would fire once, at trace time, and
record a single tracer. `jax.debug.callback(..., ordered=True)` is what
actually runs every step, in step order.

The wrapper is placed **outermost**, so one recorder yields both quantities:
its input is the estimate DP-SGD released *before* any projection, and
`params + increment` is the point the whole stack produced *after* it. The cell
below records the control and asserts the result is bit-identical to section
3's — which is the evidence that the instrumentation is inert.

In [ ]:
def recording(optimizer, released, iterate):
    """`optimizer`, plus two host-side records. Changes nothing it wraps."""

    def update_fn(estimate, state, params=None):
        # ordered=True, and a callback rather than a bare append: the step is
        # jitted with the optimizer inside it, so an append would fire at
        # trace time only. See the note above.
        jax.debug.callback(released.append, l1_norm_traced(estimate),
                           ordered=True)
        increment, new_state = optimizer.update(estimate, state, params)
        jax.debug.callback(iterate.append,
                           l1_norm_traced(pytree.add(params, increment)),
                           ordered=True)
        return increment, new_state

    return updates.Optimizer(optimizer.init, update_fn)


released_control, iterate_control = [], []
t0 = time.time()
recorded_params = run_arm(optimizer=recording(
    updates.sgd(LEARNING_RATE), released_control, iterate_control))

# The instrumentation is inert, and this is the check rather than the promise.
assert jax.tree_util.tree_all(jax.tree_util.tree_map(
    lambda a, b: bool((a == b).all()), control_params, recorded_params))
assert len(released_control) == STEPS == len(iterate_control)

RELEASED = np.array([float(v) for v in released_control])
TRAJECTORY_CONTROL = np.array([float(v) for v in iterate_control])

tail = RELEASED[-200:]
print(f"recorded control: {time.time() - t0:.1f}s, "
      f"{len(released_control):,} callbacks, "
      f"parameters bit-identical to section 3's control\n")
print(f"||g~||_1 over the whole run: min {RELEASED.min():.4f}, "
      f"max {RELEASED.max():.4f}")
print(f"||g~||_1 over the last 200 steps: "
      f"mean {tail.mean():.4f} +- {tail.std():.4f} (sd)")
print()
print(f"paper radius   C*sqrt(d) = {RADIUS_PAPER:.2f}  ->  "
      f"{RADIUS_PAPER / tail.mean():.0f}x the released norm "
      f"({np.log10(RADIUS_PAPER / tail.mean()):.2f} decades)")
print(f"forced radius            = {RADIUS_BINDING}  ->  "
      f"{tail.mean() / RADIUS_BINDING:.1f}x below the released norm")

The same thing over the run, on a log axis because the two reference radii are
nearly four decades apart. One series, so the title names it and there is no
legend box; the two radii are muted dotted lines, annotated in place.

In [ ]:
fig, axis = plt.subplots(figsize=(9, 5))

axis.plot(np.arange(1, STEPS + 1), RELEASED, color=SERIES["control"],
          linewidth=1.0, alpha=0.85)

for radius, label in ((RADIUS_PAPER, f"paper radius C*sqrt(d) = "
                                     f"{RADIUS_PAPER:.2f}"),
                      (RADIUS_BINDING, f"forced radius = {RADIUS_BINDING}")):
    axis.axhline(radius, color=MUTED, linestyle=":", alpha=0.9)
    axis.annotate(label, xy=(STEPS, radius), xytext=(-4, 5),
                  textcoords="offset points", ha="right", color=MUTED,
                  fontsize=9)

axis.set_yscale("log")
axis.set_xlabel("optimizer step")
axis.set_ylabel(r"$\|\tilde{g}\|_1$  (released estimate)")
axis.set_xlim(0, STEPS)
axis.set_title("What DP-SGD releases, against the two projection radii "
               "(control arm)")
axis.grid(alpha=0.5)
fig.tight_layout()
plt.show()

### Reading the released norms

The released estimate settles at `‖g̃‖₁ ≈ 0.41 ± 0.06` and never once in 2,000
steps exceeds 1.37. The paper's radius is 31.62. **The ball is roughly 77 times
too wide to ever touch what this mechanism releases — 1.9 orders of magnitude
of headroom — so the projection at that radius is arithmetically the identity
map.** That is the whole explanation of section 5's estimate-side rows, and it
is a measurement rather than an inference.

The reason the headroom is this large is Lemma 3.1's premise, which this model
does not satisfy. `C·√d` is the right radius when per-example gradients are
`s`-sparse and the released sum is dense with noise; the projection then strips
noise while the sparse signal survives. Here `d = 40`, every feature is dense
after frequency encoding and standardization, and the per-example gradient of a
logistic model is `(σ(x·w) − y)·x`, which is as dense as `x`. There is no
sparsity for the projection to exploit, and `C·√d` is sized for a
worst case that never arrives.

`RADIUS_BINDING = 0.19` is the answer to that: it sits about 2.1× *below* the
typical released norm, so the projection is forced to do something. Section 5
reports what forcing it buys, which is almost nothing.

## 5. The six arms, one table

Six arms. Every one is the same `run_arm` call, the same seeds, the same σ and
therefore the same ε; the only thing that varies is the optimizer object.

| arm | optimizer |
|---|---|
| control | `updates.sgd(0.1)` |
| iterate-projected, wide | `l1_projected(sgd, 1e6)` |
| iterate-projected, binding | `l1_projected(sgd, 2.6301)` |
| estimate-projected, wide | `l1_projected_estimate(sgd, 1e6)` |
| estimate-projected, paper | `l1_projected_estimate(sgd, 31.62)` |
| estimate-projected, forced | `l1_projected_estimate(sgd, 0.19)` |

The two wide arms are the identity check: at radius 1e6 neither ball can bind,
so both must reproduce the control exactly, and a difference would mean a
wrapper is doing something other than projecting.

A **constant predictor** closes the table as the reference row — it predicts
the training base rate for every record. Three conventions apply to it, all
notebook 01's:

- its **PR-AUC is entered as the test base rate**. Every score is tied, so an
  average precision computed over the sorted array reports the arrival order of
  ties rather than a ranking; the printed value below shows what that artifact
  reads, and it is why PR #21 and PR #24 quote 0.2520 and 0.2522 for the same
  row.
- its **precision is `0.0000` because it is undefined** — its output equals the
  threshold and the rule is strictly-greater, so it predicts no clicks at all.
- its **`‖θ‖₁` and non-zero count are entered as zero**: it has no parameters.

In [ ]:
ARMS = [
    ("iterate-projected,  r = 1e6", l1_projected, RADIUS_WIDE),
    (f"iterate-projected,  r = {RADIUS_ITER:.4f}", l1_projected, RADIUS_ITER),
    ("estimate-projected, r = 1e6", l1_projected_estimate, RADIUS_WIDE),
    (f"estimate-projected, r = {RADIUS_PAPER:.2f}", l1_projected_estimate,
     RADIUS_PAPER),
    (f"estimate-projected, r = {RADIUS_BINDING}", l1_projected_estimate,
     RADIUS_BINDING),
]

t0 = time.time()
RESULTS = {"DP-SGD (control)": CONTROL}
for name, wrap, radius in ARMS:
    RESULTS[name] = evaluate(
        run_arm(optimizer=wrap(updates.sgd(LEARNING_RATE), radius)))

constant_probs = np.full(len(Y_TEST), train_base_rate)
tied_pr_auc = pr_curve(constant_probs, Y_TEST)[2]
CONSTANT = score(constant_probs, None)._replace(pr_auc=test_base_rate)
print(f"five arms in {time.time() - t0:.1f}s\n")

header = (f"{'arm':<31}{'log-loss':>9}{'NE':>8}{'ECE':>8}{'cal':>8}"
          f"{'PR-AUC':>8}{'prec':>7}{'recall':>8}{'||w||_1':>9}{'non-zero':>10}")
print(header)
print("-" * len(header))
for name, r in list(RESULTS.items()) + [("constant (base rate)", CONSTANT)]:
    if name == "constant (base rate)":
        print("-" * len(header))
    print(f"{name:<31}{r.loss:9.4f}{r.ne:8.4f}{r.ece:8.4f}{r.cal:8.4f}"
          f"{r.pr_auc:8.4f}{r.precision:7.4f}{r.recall:8.4f}{r.l1:9.4f}"
          f"{r.nnz:>6} of {D_TOTAL}")

print(f"\nthreshold: probs > {THRESHOLD:.4f} (the training base rate)")
print(f"constant-row PR-AUC entered as the test base rate "
      f"{test_base_rate:.4f}; average precision over the tied scores would "
      f"read {tied_pr_auc:.4f}, which is the tie artifact, not a ranking.")

### Reading the table

**The two wide arms reproduce the control to every printed digit**, including
`‖θ‖₁` to four decimals and all 40 parameters non-zero. Both wrappers are
inert when the ball cannot bind, which is the identity check passing.

**The iterate side binds, and it costs a little.** At `r = 2.6301` — half the
control's norm, chosen to bind — log-loss rises 0.5142 → 0.5182, PR-AUC falls
0.4475 → 0.4351, and **7 of the 40 parameters are driven to exactly zero**. The
ℓ₁ projection soft-thresholds, so exact zeros are what it produces rather than
small values; that is the constraint doing its job. Calibration moves the other
way — the calibration ratio goes 1.0298 → 0.9884, from over-predicting clicks
by 3% to under-predicting by 1% — while ECE roughly doubles, so the shrunk
model is not better calibrated, it is differently miscalibrated. Recall
*improves* (0.6333 → 0.6528) at the cost of precision, which is what shrinking
scores toward the threshold does to a fixed cut, not a real gain.

**The estimate side is a no-op, exactly as ADR-0015 predicts.** At the paper's
`C·√d = 31.62` it reproduces the control digit for digit, for the reason
section 4 measured: the ball is 77× wider than anything the mechanism releases.
Forced 2.1× *below* the released norm at `r = 0.19` — a radius with no
principle behind it, chosen only to make the projection bind — it moves
log-loss by **0.0001**, from 0.5142 to 0.5141, and leaves all 40 parameters
non-zero. Projecting the estimate rescales the direction the rule descends
along; it does not constrain where the iterates end up, so `‖θ‖₁` lands at
4.9642 rather than anywhere near 0.19.

Reported per ADR-0009's pattern: the assumption behind the transform is stated
(sparse per-example gradients), it is measured (section 4), and the finding is
that **it does not hold on this problem** — so the transform that depends on it
does nothing here. That is a property of dense tabular logistic regression, not
a defect in the transform.

Both PR tables reproduce digit-for-digit, which is the secondary result: the
loader, the accountant and both transforms have not drifted since PRs #21
and #24.

### The table checks itself

The paragraph above claims the two PR tables reproduce digit-for-digit. Prose
cannot enforce that, and the staleness blockquote at the end of this notebook
only warns that prose *may* be wrong after a change — it cannot detect one.
This cell can, so the claim is written as assertions instead.

Every number below is pinned at **the precision the table above prints**, which
is the precision at which PRs #21 and #24 published them: `f"{x:.4f}"` compared
against the published string, not a float tolerance invented here. The
comparison is on released numbers only — the metrics, `‖θ‖₁` and the non-zero
count — and never on a wrapper's internals, which are free to change.

If the loader, the accountant or either transform drifts, this cell fails and
the notebook stops, rather than quietly printing different numbers under prose
that still claims the old ones.

In [ ]:
COLUMNS = ("log-loss", "NE", "ECE", "cal", "PR-AUC", "prec", "recall",
           "||w||_1", "non-zero")


def printed(e: Eval) -> tuple[str, ...]:
    """One arm as the nine strings the table above printed."""
    return (f"{e.loss:.4f}", f"{e.ne:.4f}", f"{e.ece:.4f}", f"{e.cal:.4f}",
            f"{e.pr_auc:.4f}", f"{e.precision:.4f}", f"{e.recall:.4f}",
            f"{e.l1:.4f}", f"{e.nnz}")


# PR #21's and PR #24's published rows, at their published precision.
PR_CONTROL = ("0.5142", "0.9108", "0.0110", "1.0298", "0.4475", "0.3944",
              "0.6333", "5.2602", "40")
PR_ESTIMATE_BINDING = ("0.5141", "0.9107", "0.0108", "1.0293", "0.4464",
                       "0.3948", "0.6320", "4.9642", "40")

failures = []


def expect(name, got, want, columns=COLUMNS):
    """Record every mismatched column, so a failure reports all of them."""
    for column, a, b in zip(columns, got, want):
        if a != b:
            failures.append(f"{name}: {column} printed {a}, expected {b}")


# The control row, all nine columns (PR #21).
expect("control", printed(CONTROL), PR_CONTROL)

# The three non-binding arms must equal the control in all nine columns: two
# balls that cannot bind, and the paper's radius, which section 4 shows is 77x
# too wide to touch what the mechanism releases.
for arm in ("iterate-projected,  r = 1e6",
            "estimate-projected, r = 1e6",
            f"estimate-projected, r = {RADIUS_PAPER:.2f}"):
    expect(f"{arm} vs control", printed(RESULTS[arm]), printed(CONTROL))

# The binding iterate row: the three columns the constraint moves (PR #21).
expect(f"iterate-projected,  r = {RADIUS_ITER:.4f}",
       (f"{RESULTS[f'iterate-projected,  r = {RADIUS_ITER:.4f}'].loss:.4f}",
        f"{RESULTS[f'iterate-projected,  r = {RADIUS_ITER:.4f}'].l1:.4f}",
        f"{RESULTS[f'iterate-projected,  r = {RADIUS_ITER:.4f}'].nnz}"),
       ("0.5182", "2.6301", "33"),
       columns=("log-loss", "||w||_1", "non-zero"))

# The binding estimate row, all nine columns (PR #24).
expect(f"estimate-projected, r = {RADIUS_BINDING}",
       printed(RESULTS[f"estimate-projected, r = {RADIUS_BINDING}"]),
       PR_ESTIMATE_BINDING)

if failures:
    raise AssertionError(
        "the published tables no longer reproduce:\n  "
        + "\n  ".join(failures))

print("PR #21 and PR #24 reproduce digit-for-digit, at printed precision:")
print(f"  control row pinned on all {len(COLUMNS)} columns")
print("  iterate-projected r = 1e6, estimate-projected r = 1e6 and "
      f"estimate-projected r = {RADIUS_PAPER:.2f}")
print(f"    each identical to the control on all {len(COLUMNS)} columns")
print(f"  iterate-projected  r = {RADIUS_ITER:.4f} pinned on "
      "log-loss, ||w||_1 and non-zero")
print(f"  estimate-projected r = {RADIUS_BINDING} pinned on all "
      f"{len(COLUMNS)} columns")

## 6. What the iterate-side constraint costs, over radius

Section 5 shows one binding radius. The shape of the cost needs several, so the
iterate-side arm is swept over multiples of the control's norm.

**The unconstrained point is drawn as a reference line, not as a marker.** Its
radius is 1e6, five decades from the rest of the sweep; plotting it would
compress every informative point into the left edge of a log axis. Its value is
in the printed table below, and it appears in both panels as a muted annotated
horizontal line — which is also what it *is*: the level the curve is
approaching.

In [ ]:
SWEEP_MULTIPLES = (0.25, 0.5, 0.75, 1.0, 1.5, 2.0)

t0 = time.time()
SWEEP = []
for multiple in SWEEP_MULTIPLES:
    radius = multiple * CONTROL.l1
    SWEEP.append((radius, evaluate(
        run_arm(optimizer=l1_projected(updates.sgd(LEARNING_RATE), radius)))))
print(f"sweep of {len(SWEEP)} runs in {time.time() - t0:.1f}s\n")

SWEEP_RADII = np.array([r for r, _ in SWEEP])
SWEEP_LOSS = np.array([e.loss for _, e in SWEEP])
SWEEP_NNZ = np.array([e.nnz for _, e in SWEEP])
WIDE = RESULTS["iterate-projected,  r = 1e6"]

print(f"{'radius':>10}{'x control':>11}{'log-loss':>10}{'PR-AUC':>9}"
      f"{'||w||_1':>10}{'non-zero':>11}")
print("-" * 61)
for multiple, (radius, e) in zip(SWEEP_MULTIPLES, SWEEP):
    print(f"{radius:10.4f}{multiple:11.2f}{e.loss:10.4f}{e.pr_auc:9.4f}"
          f"{e.l1:10.4f}{e.nnz:>7} of {D_TOTAL}")
print(f"{'1e6':>10}{'-':>11}{WIDE.loss:10.4f}{WIDE.pr_auc:9.4f}"
      f"{WIDE.l1:10.4f}{WIDE.nnz:>7} of {D_TOTAL}")

The two columns that move, on their own scales. Log-x, because the radii span a
factor of eight and the interesting behaviour is at the small end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for axis, values, reference, ylabel, title in (
    (axes[0], SWEEP_LOSS, WIDE.loss, "test log-loss",
     "What the ball costs"),
    (axes[1], SWEEP_NNZ, WIDE.nnz, f"non-zero parameters (of {D_TOTAL})",
     "What the ball zeroes"),
):
    axis.plot(SWEEP_RADII, values, marker="o",
              color=SERIES["iterate-projected"])
    axis.axhline(reference, color=MUTED, linestyle=":", alpha=0.9)
    axis.annotate("unconstrained (r = 1e6)", xy=(SWEEP_RADII[0], reference),
                  xytext=(0, -13), textcoords="offset points", color=MUTED,
                  fontsize=9)
    axis.set_xscale("log")
    axis.set_xlabel(r"$\ell_1$ ball radius")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(alpha=0.5)

fig.suptitle("Iterate-side projection, swept over radius", color=INK)
fig.tight_layout()
plt.show()

### Reading the sweep

The cost is flat until the ball is genuinely tight and then rises sharply. At
`r ≥ 3.9451` (0.75× the control's norm) log-loss is 0.5143 against the
unconstrained 0.5142 and 39 of 40 parameters survive — the constraint is
nominally active but costs nothing measurable. At 2.6301 it costs 0.0040 and 7
parameters. At 1.3150 it costs 0.0287 and **32 of 40 parameters**, leaving 8.

Two details worth naming. At exactly 1.0× the control's norm the projection
still binds slightly — `‖θ‖₁` lands at 5.2476 rather than 5.2602 — because the
ball constrains the whole trajectory, not just the endpoint, and a path that
would have wandered outside it is bent. And above 1.5× the returned norm is the
control's 5.2602 to four decimals, so the sweep meets the unconstrained line
rather than approaching it asymptotically.

This is a shrinkage curve, and it is the ordinary one: the ℓ₁ ball buys
sparsity and pays in fit, with a comfortable region where it buys a little
sparsity for nearly nothing. Nothing about it is specific to privacy — which is
the point. The projection is post-processing, so **what it costs is a
statement about constrained optimization, not about the mechanism**.

Last, the same constraint seen over time rather than at the end. The recorder
from section 4 gives `‖θ_t‖₁` at every step for three arms: the control, the
iterate-projected arm at the binding radius, and the estimate-projected arm at
the forced radius. Two runs, since the control's trajectory is already
recorded.

The third series is one addition beyond a straight control-versus-binding-arm
picture. It costs a run that section 5 already needed and it makes the duality
visible in one frame: the two wrappers take the same radius argument and do
categorically different things to the trajectory.

In [ ]:
t0 = time.time()
# Only the iterate stream is wanted here; section 4 already has the released
# one for the control, and it is the same mechanism in every arm.
iterate_iter, iterate_est = [], []
run_arm(optimizer=recording(
    l1_projected(updates.sgd(LEARNING_RATE), RADIUS_ITER), [], iterate_iter))
run_arm(optimizer=recording(
    l1_projected_estimate(updates.sgd(LEARNING_RATE), RADIUS_BINDING),
    [], iterate_est))
print(f"two recorded runs in {time.time() - t0:.1f}s\n")

TRAJECTORY = {
    "control": TRAJECTORY_CONTROL,
    "iterate-projected": np.array([float(v) for v in iterate_iter]),
    "estimate-projected": np.array([float(v) for v in iterate_est]),
}
steps_axis = np.arange(1, STEPS + 1)

fig, axis = plt.subplots(figsize=(9, 5))
for name, label in (
    ("control", "control (unprojected)"),
    ("iterate-projected", f"iterate-projected, r = {RADIUS_ITER:.4f}"),
    ("estimate-projected",
     f"estimate-projected, r = {RADIUS_BINDING}"),
):
    axis.plot(steps_axis, TRAJECTORY[name], color=SERIES[name],
              linewidth=1.6, label=f"{label}   ends at "
                                   f"{TRAJECTORY[name][-1]:.4f}")

axis.axhline(RADIUS_ITER, color=MUTED, linestyle=":", alpha=0.9)
axis.annotate(f"the ball, r = {RADIUS_ITER:.4f}", xy=(STEPS, RADIUS_ITER),
              xytext=(-4, 6), textcoords="offset points", ha="right",
              color=MUTED, fontsize=9)

first_bind = int(np.argmax(TRAJECTORY["iterate-projected"]
                           >= RADIUS_ITER - 1e-4)) + 1
axis.set_xlabel("optimizer step")
axis.set_ylabel(r"$\|\theta_t\|_1$")
axis.set_xlim(0, STEPS)
axis.set_title("Where the iterates go, with and without each projection")
axis.legend(loc="lower right")
axis.grid(alpha=0.5)
fig.tight_layout()
plt.show()

print(f"the iterate-projected arm first reaches its ball at step "
      f"{first_bind} and is pinned to it for the remaining "
      f"{STEPS - first_bind:,} steps")

### Reading the trajectory

The iterate-projected arm climbs with the control for the first 58 steps, hits
the ball at step 59, and is **pinned to it exactly for the remaining 1,941** —
the orange
line is flat at 2.6301 because that is what constraining the iterates means.
Every subsequent step is a descent step followed by a projection back onto the
boundary.

The control and the estimate-projected arm climb past it and keep going,
separating from each other only slightly: 5.2602 against 4.9642 at the end. The
aqua line is the visual form of section 5's finding — projecting the *estimate*
at a radius 2.1× below the released norm rescales each step a little, and the
trajectory it produces is a slightly damped copy of the control's rather than a
constrained one. **Same wrapper argument, same seam, categorically different
object constrained.**

Note also that the control's own norm is still rising at step 2,000. Nothing
here has converged, which is section 2's point about a reproduction budget
restated as a picture: these are 2,000-step trajectories because that is what
the PRs ran, and a longer run would move every `‖θ‖₁` in this figure.

## What this shows, and what it does not

> **Stale until re-run.** Every number quoted below is read off library code at
> commit `6e84325` ("Add the l1 projection of the released estimate"), executed
> in this file. The transforms, the loader and the accountant are all under
> active development; if any of them moves, the numbers in the prose are wrong
> until the notebook is re-executed and the prose is re-read against it.

**The comparison.** DP-SGD on Criteo at ε = 3, δ = 1e-6 over 2,000 steps, against
the same run with each of the two ℓ₁ projections inserted at the optimizer
seam. Every arm shares an initialization, both random streams, `C`, `η`, `L`,
σ and therefore ε; the optimizer object is the only difference. Both PR #21's
and PR #24's tables reproduce digit-for-digit.

**The finding is two-sided.** Constraining the *iterates* does what constraining
iterates does: at half the control's norm it costs 0.0040 log-loss and 0.0124
PR-AUC and zeroes 7 of 40 parameters, and section 6's sweep traces the whole
shrinkage curve, flat until the ball is tight and then steep. Constraining the
*estimate* does nothing at any radius with a principle behind it — identical to
the control at the paper's `C·√d = 31.62`, and worth 0.0001 log-loss when
forced 2.1× below what the mechanism actually releases. Section 4 measures the
reason: `‖g̃‖₁ ≈ 0.41`, against a prescribed ball of 31.62, so the projection is
the identity map with 1.9 decades to spare. Ghazi et al.'s Lemma 3.1 wants
sparse per-example gradients; a dense 40-feature logistic model has none, so
the transform that rests on that premise has nothing to strip. That is a
negative result about this problem, reported as one — the transform is not at
fault and neither is the paper.

Not shown, and not claimed:

- **Neither wrapper makes a privacy claim.** Both are post-processing applied
  after the mechanism releases its estimate, so every arm carries DP-SGD's own
  ε and no other. **That ε is RDP's** (ADR-0011); PLD would report a smaller one
  for the same run, and the method is held fixed at `"rdp"` throughout, which is
  the property a comparison needs. The full statement lives here; sections 2
  and 4 carry the in-place reminders, deliberately — a reader who stops at the
  wrapper should not have to reach this list to learn it claims nothing.
- **This is Ghazi et al. 2024's projection *step*, not their *mechanism*.** The
  library ports the projection at the optimizer seam (ADR-0014, ADR-0015); the
  paper's standalone sparse-gradient mechanism is deliberately not ported, and
  nothing here evaluates it. A negative result about the step is not a negative
  result about the paper.
- **One seed, no error bars.** Every arm runs at one initialization and one
  pair of streams. The differences reported on the estimate side — 0.0001 in
  log-loss — are well inside what a second seed would move, and are quoted only
  to say "indistinguishable from the control", never as a measured effect. The
  iterate-side differences are large enough to survive that caveat; they have
  still not been repeated.
- **2,000 steps, chosen for reproduction and not for convergence.** Section 6's
  trajectory shows `‖θ‖₁` still climbing at the last step. Every radius in this
  notebook is scaled off a control that has not converged, so the sweep's
  x-axis would move under a longer budget.
- **One binding radius per side, and one of them is arbitrary.** `RADIUS_ITER`
  is read off the control specifically to bind, and `RADIUS_BINDING = 0.19` has
  no justification beyond forcing the estimate-side projection to do something.
  Neither is a recommendation.
- **Preprocessing is unaccounted** (ADR-0008), the ε covers the training loop
  only, and the test evaluation sits outside it by design.
- **No SpiderBoost.** The comparison against Private SpiderBoost is issue #11's
  notebook.
- **One dataset, one model.** Logistic regression on Criteo, 39 dense features.
  The estimate-side no-op is a statement about *this* gradient geometry; a
  genuinely sparse problem is where that transform is supposed to earn its
  place, and this notebook does not build one.